# Classification Training — Adım Adım

`01_train_classification.py` scriptinin tam olarak ne yaptığını, SLURM'a göndermeden önce burada hücre hücre izleyebilirsin.

**Sıra:**
1. Config yükle
2. Parquet yollarını resolve et
3. Dataset + DataLoader kur
4. İlk batch sanity check
5. Model kur (DynEdge + BinaryClassificationTask)
6. Callbacks + Trainer kur
7. `trainer.fit()` başlat

---
## 0. Imports & path setup

In [ ]:
import sys, os, math, importlib.util

SCRIPT_DIR = "/project/def-nahee/kbas/graphnet/examples/08_pone"
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

import yaml
import torch
import pytorch_lightning as pl

from graphnet.data.dataloader import DataLoader
from graphnet.data.dataset import EnsembleDataset
from graphnet.data.dataset.parquet.parquet_dataset import ParquetDataset
from graphnet.models.data_representation import KNNGraph, NodesAsPulses
from graphnet.models.detector.pone import PONE
from graphnet.models.gnn import DynEdge
from graphnet.models.standard_model import StandardModel
from graphnet.models.task.classification import BinaryClassificationTask
from graphnet.training.callbacks import GraphnetEarlyStopping, PiecewiseLinearLR
from graphnet.training.loss_functions import BinaryCrossEntropyLoss

from utils import (
    EpochCSVLogger,
    EpochTimeLogger,
    _EpochContextCallback,
    extract_field,
    install_logging_filters,
)

print('Imports OK')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

---
## 1. Config yükle

Script `python 01_train_classification.py -c config.yaml` şeklinde çalışıyor.  
Burada `-c` argümanının yerini aşağıdaki hücre alıyor.

In [ ]:
CONFIG_PATH = "/project/def-nahee/kbas/graphnet/examples/08_pone/configs/classification/exp001.yml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print(yaml.dump(cfg, default_flow_style=False))

In [ ]:
# Spring2026MC için mc override:
# cfg["mc"] = "Spring2026MC"

print('mc      :', cfg['mc'])
print('geometry:', cfg['geometry'])
print('flavors :', cfg.get('flavors', ['Muon','Electron','Tau','NC']))

---
## 2. `resolve_paths` — Parquet yollarını bul

`paths.py` dosyası `importlib` ile runtime'da yükleniyor.  
Config'deki `mc` + `geometry` kombinasyonuna göre her flavor için `train/val/test` parquet yolları ve `percentiles_csv` döndürülüyor.

In [ ]:
PATHS_PY    = "/project/def-nahee/kbas/Graphnet-Applications/Metadata/paths.py"
ALL_FLAVORS = ['Muon', 'Electron', 'Tau', 'NC']

PARQUET_TABLE = {
    '340StringMC':  'STRING340MC_PARQUET',
    'Spring2026MC': 'SPRING2026MC_PARQUET',
}
PARQUET_MIXED_TABLE = {
    '340StringMC':  'STRING340MC_PARQUET_MIXED',
    'Spring2026MC': 'SPRING2026MC_PARQUET_MIXED',
}

def resolve_paths(cfg):
    mc       = cfg['mc']
    geometry = cfg['geometry']
    flavors  = cfg.get('flavors', ALL_FLAVORS)

    spec = importlib.util.spec_from_file_location('paths', PATHS_PY)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    parquet_table = getattr(mod, PARQUET_TABLE[mc])
    parquet_mixed = getattr(mod, PARQUET_MIXED_TABLE[mc])

    per_flavor = {}
    for flavor in flavors:
        entry = parquet_table.get(geometry, {}).get(flavor, {})
        for key in ('train', 'val', 'test'):
            if not entry.get(key):
                raise ValueError(f"{PARQUET_TABLE[mc]}['{geometry}']['{flavor}']['{key}'] is None")
        per_flavor[flavor] = entry
        print(f'[{flavor}] train: {entry["train"]}')

    percentiles_csv = parquet_mixed.get(geometry, {}).get('percentiles_csv')
    if not percentiles_csv:
        raise ValueError('percentiles_csv is None — compute_mixed_percentiles.py calistir')
    print(f'\npercentiles_csv: {percentiles_csv}')

    return per_flavor, percentiles_csv

In [ ]:
per_flavor, percentiles_csv = resolve_paths(cfg)

print('\n--- per_flavor splits ---')
for flv, paths in per_flavor.items():
    print(f'  {flv}: {list(paths.keys())}')

---
## 3. Dataset + DataLoader kur

### 3a. `KNNGraph` data representation

Her event bir **graph**'a dönüştürülüyor:
- **Node**: Her pulse → `[pmt_x, pmt_y, pmt_z, dom_time, charge]`
- **Edge**: Her node'un k-nearest-neighbour'ları arasında
- `PONE` detector: feature normalizasyonu için `percentiles_csv` kullanıyor

In [ ]:
features = cfg['data']['features']
print('Features     :', features)
print('nb_neighbours:', cfg['model']['nb_neighbours'])

data_representation = KNNGraph(
    detector=PONE(percentiles_csv=percentiles_csv, selected_features=features),
    node_definition=NodesAsPulses(),
    nb_nearest_neighbours=cfg['model']['nb_neighbours'],
    distance_as_edge_feature=False,
)
print('KNNGraph kuruldu.')

### 3b. `ParquetDataset` ve `EnsembleDataset`

Her flavor için ayrı `ParquetDataset` oluşturuluyor.  
`EnsembleDataset` bunları tek bir dataset gibi birleştiriyor — train/val/test için ayrı ayrı.

In [ ]:
truth_all   = cfg['data']['truth_all']
pulsemaps   = cfg['data']['pulsemaps']
truth_table = cfg['data']['truth_table']

def _make_dataset(path):
    return ParquetDataset(
        path=path,
        pulsemaps=pulsemaps,
        truth_table=truth_table,
        features=features,
        truth=truth_all,
        data_representation=data_representation,
    )

train_ds = EnsembleDataset([_make_dataset(e['train']) for e in per_flavor.values()])
val_ds   = EnsembleDataset([_make_dataset(e['val'])   for e in per_flavor.values()])
test_ds  = EnsembleDataset([_make_dataset(e['test'])  for e in per_flavor.values()])

print(f'train_ds: {len(train_ds):,} event')
print(f'val_ds  : {len(val_ds):,} event')
print(f'test_ds : {len(test_ds):,} event')

### 3c. DataLoader

- `spawn` multiprocessing — SLURM'da güvenli
- `pin_memory=True` — CPU→GPU transfer'i hızlandırır
- `persistent_workers=True` — worker process'leri epoch'lar arası canlı tutar
- `drop_last=True` (sadece train) — gradient accumulation ile tutarlı batch sayısı için

In [ ]:
tcfg = cfg['training']

def _make_loader(ds, shuffle, drop_last=False):
    return DataLoader(
        ds,
        batch_size=tcfg['batch_size'],
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=tcfg['num_workers'],
        multiprocessing_context=tcfg.get('multiprocessing_context', 'spawn'),
        persistent_workers=True,
        pin_memory=tcfg.get('pin_memory', True),
    )

train_loader = _make_loader(train_ds, shuffle=True,  drop_last=True)
val_loader   = _make_loader(val_ds,   shuffle=False)
test_loader  = _make_loader(test_ds,  shuffle=False)

print(f'batch_size : {tcfg["batch_size"]}')
print(f'num_workers: {tcfg["num_workers"]}')
print(f'train_loader: {len(train_loader)} batch')
print(f'val_loader  : {len(val_loader)} batch')
print(f'test_loader : {len(test_loader)} batch')

---
## 4. Sanity check — İlk batch

Script burada `next(iter(train_loader))` yapıp `is_track` labellarını kontrol ediyor.
- `is_track=1` → track (muon)
- `is_track=0` → cascade (electron, tau, NC)
- `mean ≈ 0.25` beklenir (4 flavor'dan 1'i track)

In [ ]:
print('Ilk batch cekiliyor...')
b0 = next(iter(train_loader))
print('Batch tipi :', type(b0))
print('num_graphs :', b0.num_graphs if hasattr(b0, 'num_graphs') else len(b0))

In [ ]:
labels = extract_field(b0, 'is_track').detach().cpu().view(-1)

print(f'is_track shape: {labels.shape}')
print(f'min={labels.min().item():.0f}  max={labels.max().item():.0f}  mean={labels.mean().item():.3f}')
print(f'\ntrack   (1): {(labels==1).sum().item()} event')
print(f'cascade (0): {(labels==0).sum().item()} event')

In [ ]:
# Graph yapısını incele
if hasattr(b0, 'x'):
    print('node feature matrix (x) shape:', b0.x.shape)
    print('feature names                 :', features)
    print('ilk 3 node:')
    print(b0.x[:3])
    print('\nedge_index shape:', b0.edge_index.shape)

---
## 5. Model kur

### 5a. LR schedule hesapla

`PiecewiseLinearLR` **step** bazlı çalışıyor:

```
step 0           warmup_steps       total_steps
  |                   |                  |
base_lr  ──────►  peak_lr  ──────────►  base_lr
```

In [ ]:
acc = tcfg['accumulate_grad_batches']
steps_per_epoch_optimizer = math.ceil(len(train_loader) / acc)
total_steps  = steps_per_epoch_optimizer * tcfg['max_epochs']
warmup_steps = max(1, int(tcfg.get('warmup_fraction', 0.5) * steps_per_epoch_optimizer))

print(f'train batches           : {len(train_loader)}')
print(f'accumulate_grad_batches : {acc}')
print(f'steps_per_epoch         : {steps_per_epoch_optimizer}')
print(f'max_epochs              : {tcfg["max_epochs"]}')
print(f'total_steps             : {total_steps}')
print(f'warmup_steps            : {warmup_steps}  (warmup_fraction={tcfg.get("warmup_fraction",0.5)})')
print(f'base_lr → peak_lr       : {tcfg["base_lr"]} → {tcfg["peak_lr"]}')

### 5b. DynEdge backbone

Her node komşularından mesaj alıyor (dynamic edge convolution).  
`global_pooling_schemes` ile tüm node'lar event-level representation'a dönüştürülüyor.

In [ ]:
mcfg = cfg['model']

backbone = DynEdge(
    nb_inputs=len(features),
    nb_neighbours=mcfg['nb_neighbours'],
    global_pooling_schemes=mcfg['global_pooling_schemes'],
    add_global_variables_after_pooling=mcfg.get('add_global_variables_after_pooling', True),
    add_norm_layer=mcfg.get('add_norm_layer', False),
    skip_readout=mcfg.get('skip_readout', False),
)

print(f'nb_inputs  : {len(features)}')
print(f'nb_outputs : {backbone.nb_outputs}')
print(f'pooling    : {mcfg["global_pooling_schemes"]}')
print(f'\nDynEdge parametre sayisi: {sum(p.numel() for p in backbone.parameters()):,}')

### 5c. BinaryClassificationTask

Backbone çıktısını alıp `track_score ∈ [0,1]` üretiyor.  
Loss: `BinaryCrossEntropyLoss`

In [ ]:
task = BinaryClassificationTask(
    hidden_size=backbone.nb_outputs,
    loss_function=BinaryCrossEntropyLoss(),
    target_labels=['is_track'],
    prediction_labels=['track_score'],
)
print('task kuruldu — target: is_track  output: track_score')

### 5d. StandardModel — her şeyi birleştir

In [ ]:
model = StandardModel(
    tasks=task,
    data_representation=data_representation,
    backbone=backbone,
    optimizer_class=torch.optim.Adam,
    optimizer_kwargs={'lr': tcfg['base_lr']},
    scheduler_class=PiecewiseLinearLR,
    scheduler_kwargs={
        'milestones': [0, warmup_steps, total_steps],
        'factors':    [1.0, tcfg['peak_lr'] / tcfg['base_lr'], 1.0],
    },
    scheduler_config={'interval': 'step'},
)

print(f'Toplam parametre: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Forward pass ile sekil kontrol
model.eval()
with torch.no_grad():
    preds = model(b0)

print('Forward pass OK')
print('Output sayisi (task sayisi) :', len(preds))
print('track_score shape           :', preds[0].shape)
print('track_score ilk 5           :', preds[0][:5].view(-1).tolist())

---
## 6. Callbacks + Trainer kur

| Callback | Ne yapar |
|---|---|
| `GraphnetEarlyStopping` | `val_loss` izler, patience dolunca `best_model.pth` kaydeder ve durur |
| `EpochCSVLogger` | Her epoch: `train_loss`, `val_loss`, `lr` → `metrics.csv` |
| `EpochTimeLogger` | Her epoch: süre + RAM + GPU → `resources_and_time.csv` |
| `_EpochContextCallback` | Early stopping loglarına epoch numarası enjekte eder |

In [ ]:
install_logging_filters()

out_dir = os.path.join(cfg['output']['save_dir'], 'classification')
os.makedirs(out_dir, exist_ok=True)
print('out_dir:', out_dir)

pl.seed_everything(tcfg['seed'], workers=True)

early_stop   = GraphnetEarlyStopping(
    save_dir=out_dir,
    monitor='val_loss',
    mode='min',
    patience=tcfg['early_stopping_patience'],
    check_on_train_epoch_end=False,
    verbose=True,
)
metrics_cb   = EpochCSVLogger(out_dir, extra_keys=[])
time_cb      = EpochTimeLogger(out_dir)
epoch_ctx_cb = _EpochContextCallback()

print(f'early_stopping_patience: {tcfg["early_stopping_patience"]}')

In [ ]:
trainer = pl.Trainer(
    max_epochs=tcfg['max_epochs'],
    accelerator='gpu',
    devices=1,
    callbacks=[epoch_ctx_cb, early_stop, metrics_cb, time_cb],
    enable_checkpointing=False,
    enable_progress_bar=True,
    accumulate_grad_batches=tcfg['accumulate_grad_batches'],
)

print('Trainer hazir.')
print(f'max_epochs             : {tcfg["max_epochs"]}')
print(f'accumulate_grad_batches: {tcfg["accumulate_grad_batches"]}')

---
## 7. Training başlat

`trainer.fit()` döngüsü:
```
for epoch in range(max_epochs):
    for batch in train_loader:
        forward → loss → backward
        optimizer step (her acc batch'te bir)
        LR scheduler step
    for batch in val_loader:
        forward → val_loss  (no grad)
    EarlyStopping check → val_loss iyileştiyse best_model.pth güncelle
    EpochCSVLogger → metrics.csv'e yaz
    EpochTimeLogger → resources_and_time.csv'e yaz
```

> **Not:** Notebook'ta tam training uzun sürer.  
> Önce `limit_train_batches` + `limit_val_batches` ile 1 epoch test et, sonra SLURM'a gönder.

In [ ]:
# --- Hizli test (1 epoch, az batch) ---
# trainer_test = pl.Trainer(
#     max_epochs=1,
#     accelerator='gpu', devices=1,
#     callbacks=[epoch_ctx_cb, early_stop, metrics_cb, time_cb],
#     enable_checkpointing=False,
#     enable_progress_bar=True,
#     accumulate_grad_batches=tcfg['accumulate_grad_batches'],
#     limit_train_batches=10,
#     limit_val_batches=5,
# )
# trainer_test.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

# --- Tam training ---
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

print(f'\nbest_model : {os.path.join(out_dir, "best_model.pth")}')
print(f'metrics    : {os.path.join(out_dir, "metrics.csv")}')
print(f'resources  : {os.path.join(out_dir, "resources_and_time.csv")}')

---
## 8. Sonuçlara bak

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_csv_path = os.path.join(out_dir, 'metrics.csv')

if os.path.exists(metrics_csv_path):
    df = pd.read_csv(metrics_csv_path)
    print(df.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(df['epoch'], df['train_loss'], label='train', marker='o')
    axes[0].plot(df['epoch'], df['val_loss'],   label='val',   marker='o')
    best_rows = df[df['best_model_is_updated'] == True]
    axes[0].scatter(best_rows['epoch'], best_rows['val_loss'],
                    marker='*', s=200, color='gold', zorder=5, label='best')
    axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

    axes[1].plot(df['epoch'], df['lr'], color='green', marker='o')
    axes[1].set_xlabel('epoch'); axes[1].set_ylabel('lr')
    axes[1].set_title('Learning Rate'); axes[1].grid(True)

    plt.tight_layout()
    plt.show()
else:
    print(f'Henuz metrics.csv yok: {metrics_csv_path}')